# 1. Transcriptomic analysis of one sample

We take a single ANCA-GN core, run the
standard single-cell pipeline (normalise, embed, cluster), and annotate cell types
from marker genes. The workflow is the familiar scanpy one; the spatial layer comes in
notebook 5.

### Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt
import course_utils as cu
sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. Load one sample

> **Method note.** We analyse one core first so the steps are
fast and the biology is interpretable, then scale to all samples in notebook 3.

In [ ]:
adata = cu.load_or_build_qc()                 # all 8 samples, QC'd
SAMPLE = 'X37'                                # one ANCA-GN core
ad = adata[adata.obs['sample'] == SAMPLE].copy()
print(ad.shape, '| disease:', ad.obs['disease'][0])

## 2. Normalise, embed, cluster

> **Method note: a small panel.** With only 480
targeted genes there are too few to justify highly-variable-gene selection, so we keep
all genes. Counts are normalised to equal depth and log-transformed, then PCA, a
neighbour graph, UMAP, and Leiden, exactly as in dissociated scRNA-seq.

In [ ]:
ad.layers['counts'] = ad.X.copy()
sc.pp.normalize_total(ad)
sc.pp.log1p(ad)
# 480-gene panel: too few genes to drop any, so we skip HVG selection and use all genes
sc.pp.pca(ad, n_comps=50)
sc.pp.neighbors(ad, n_neighbors=15)
sc.tl.umap(ad)
sc.tl.leiden(ad, resolution=1.0, flavor='igraph', n_iterations=2, directed=False)
print('clusters:', ad.obs['leiden'].nunique())

In [ ]:
sc.pl.umap(ad, color=['leiden', 'total_counts'], wspace=0.3)

## 3. Annotate cell types from markers

> **Method note: marker scoring.**
`score_genes` gives each cell the mean expression of a marker set minus a random
background, so scores are comparable across sets. We average each score per Leiden
cluster and label the cluster by its highest-scoring lineage. This is transparent and
needs no reference, but a small panel makes rare or similar types harder to separate,
so always check the dotplot and the spatial plausibility.

In [ ]:
markers = cu.load_markers()
markers = {k: [g for g in v if g in ad.var_names] for k, v in markers.items()}
markers = {k: v for k, v in markers.items() if v}
for ct, genes in markers.items():
    sc.tl.score_genes(ad, genes, score_name=f'score::{ct}')
score_cols = [f'score::{ct}' for ct in markers]
cluster_score = ad.obs.groupby('leiden', observed=True)[score_cols].mean()
mapping = cluster_score.idxmax(1).str.replace('score::', '', regex=False).to_dict()
ad.obs['celltype'] = ad.obs['leiden'].map(mapping).astype('category')
mapping

Marker dotplot (do the cluster labels match the canonical markers?):

In [ ]:
# a compact marker set, one or two per lineage, for a readable dotplot
panel_markers = ['NPHS2','PODXL','CLDN1','LRP2','CUBN','UMOD','SLC12A1','SLC12A3','AQP2',
                 'SLC4A1','PECAM1','EMCN','PDGFRB','ACTA2','COL1A1','PTPRC','CD3D','LYZ','MS4A1']
panel_markers = [g for g in panel_markers if g in ad.var_names]
sc.pl.dotplot(ad, panel_markers, groupby='leiden', standard_scale='var')

## 4. Read the annotation in space

> **Method note.** A UMAP shows transcriptional
groups; the tissue plot shows whether they form sensible structures. In kidney, tubule
types should form tubule profiles, podocytes should sit in glomerular tufts, and immune
cells should be sparse and interstitial. If a label is scattered where it should be
structured, distrust it.

In [ ]:
sc.pl.umap(ad, color='celltype')
sq.pl.spatial_scatter(ad, color='celltype', shape=None, size=8, figsize=(7,7))

In [ ]:
viz = [g for g in ['NPHS2','UMOD','LRP2','PECAM1','CD3D','COL1A1'] if g in ad.var_names]
sq.pl.spatial_scatter(ad, color=viz, shape=None, size=8, ncols=3, use_raw=False)

> **Exercise.** Pick a cluster, list its top marker genes with `rank_genes_groups`,
and judge whether the automatic label is right. Would you merge or rename any clusters?

In [ ]:
# Solution: inspect one cluster's markers and decide if the auto-label is right.
cl = ad.obs['leiden'].value_counts().index[0]   # the largest cluster
sc.tl.rank_genes_groups(ad, 'leiden', groups=[cl], method='wilcoxon')
top = [ad.uns['rank_genes_groups']['names'][cl][i] for i in range(10)]
print(f'cluster {cl} -> auto label {mapping[cl]!r}; top genes:', top)

> **Exercise.** Leiden is one clustering algorithm. Run **KMeans** on the PCA embedding
(use a similar number of clusters as Leiden) and compare the two partitions with the
**Adjusted Rand Index**. Where do they disagree, and does it change which cell types you would
call?

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
k = ad.obs['leiden'].nunique()
ad.obs['kmeans'] = pd.Categorical(
    KMeans(k, n_init=10, random_state=0).fit_predict(ad.obsm['X_pca']).astype(str))
print(f'ARI(Leiden, KMeans) = {adjusted_rand_score(ad.obs["leiden"], ad.obs["kmeans"]):.3f}')
sc.pl.umap(ad, color=['leiden', 'kmeans'])

### Recap

We clustered one core and annotated it from markers. Marker scoring is
transparent but limited by the panel; notebook 2 compares it with a reference-based
classifier.